# 5.3.1: Intro to Spark/PySpark

In [1]:
import pyspark
from pyspark.sql import SparkSession

To start interacting with Spark, we need a builder object

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/28 17:17:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Download some taxi data

In [5]:
!wget -P /app/data/ https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-02-28 17:21:06--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 4.228.31.150
Connecting to github.com (github.com)|4.228.31.150|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-28T17%3A58%3A02Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-28T16%3A57%3A08Z&ske=2026-02-28T17%3A58%3A02Z&sks=b&skv=2018-11-09&sig=%2F7jQatuqYssFawQNf2%2F8Gg2SXT7qcOIk1a02CYDksiI%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjMwMjg2NywibmJmIjoxNzcyMjk5MjY3LCJwYXRo

In [8]:
!wc -l /app/data/fhvhv_tripdata_2021-01.csv.gz

508066 /app/data/fhvhv_tripdata_2021-01.csv.gz


In [10]:
df = spark.read.option('header','true').csv('/app/data/fhvhv_tripdata_2021-01.csv.gz')

In [11]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [12]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

Spark does not infer types. It treats everything as Strings

In [13]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

We will edit the schema to something more appropiate

In [14]:
from pyspark.sql import types

In [15]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [16]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('/app/data/fhvhv_tripdata_2021-01.csv.gz')

df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [17]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

We want to break down the file into smaller partitions to take advantage of Sparks distributed architecture

In [18]:
df = df.repartition(24) # Partition df in 24 partitions

Nothings is executed until we do something with the df

In [19]:
df.write.parquet('/app/data/partitioned/2021/01/') # Save the df as parquet files (-> 24 partitions)

26/02/28 17:24:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/28 17:24:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/28 17:24:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

# 5.3.2: Spark DataFrames

Let's read the parquet files we created

In [20]:
df = spark.read.parquet('/app/data/partitioned/2021/01')

In [21]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-10 17:40:01|2021-01-10 17:48:10|          97|          25|   NULL|
|           HV0005|              B02510|2021-01-08 18:19:44|2021-01-08 18:55:57|         138|         265|   NULL|
|           HV0003|              B02876|2021-01-01 16:47:20|2021-01-01 16:58:28|          50|         163|   NULL|
|           HV0005|              B02510|2021-01-15 17:50:08|2021-01-15 18:07:24|         163|          79|   NULL|
|           HV0005|              B02510|2021-01-12 17:26:40|2021-01-12 17:57:57|          47|          74|   NULL|
|           HV0003|              B02765|2021-01-05 02:00:14|2021-01-05 02:19:39|

In [22]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



Parquet stores file more efficiently by using only the amount of bytes needed for that data type (e.g 4 bytes for int)

Selecting columns

In [24]:
df.select('pickup_datetime', 'dropoff_datetime').filter(df.hvfhs_license_num == 'HV0003') 
# creates a new df with selected cols and filters for license num value


DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp]

## Actions vs transformations

**Transformations**:
- Lazy: are not executed until a trigger happens (e.g `df.show()`)
- Selecting columns, Filtering, Joins, grouping, etc...

**Actions**:
- Actually executes computation
- e.g: show, take, head, write

We can also interact with spark dfs using plain sql.

Pyspark has builtin sql functions:

In [25]:
from pyspark.sql import functions as F

In [27]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .show()
                

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|           HV0005|              B02510|2021-01-10 17:40:01|2021-01-10 17:48:10|          97|          25|   NULL| 2021-01-10|  2021-01-10|
|           HV0005|              B02510|2021-01-08 18:19:44|2021-01-08 18:55:57|         138|         265|   NULL| 2021-01-08|  2021-01-08|
|           HV0003|              B02876|2021-01-01 16:47:20|2021-01-01 16:58:28|          50|         163|   NULL| 2021-01-01|  2021-01-01|
|           HV0005|              B02510|2021-01-15 17:50:08|2021-01-15 18:07:24|         163|          79|   NULL| 2021-01-15|  2021-01-15|
|           HV0005| 

The advantage of running some transformations with pyspark is that some logic can be really complicated

In [29]:
# some crazy logic that's hard to replicate in sql
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

crazy_stuff('B02884')

's/b44'

In [30]:
# Create user defined func (udf)
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [31]:
# add column using our udf
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('baseId', crazy_stuff_udf(df.dispatching_base_num)) \
    .show()

[Stage 10:>                                                         (0 + 1) / 1]

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|baseId|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+------+
|           HV0005|              B02510|2021-01-10 17:40:01|2021-01-10 17:48:10|          97|          25|   NULL| 2021-01-10|  2021-01-10| e/9ce|
|           HV0005|              B02510|2021-01-08 18:19:44|2021-01-08 18:55:57|         138|         265|   NULL| 2021-01-08|  2021-01-08| e/9ce|
|           HV0003|              B02876|2021-01-01 16:47:20|2021-01-01 16:58:28|          50|         163|   NULL| 2021-01-01|  2021-01-01| e/b3c|
|           HV0005|              B02510|2021-01-15 17:50:08|2021-01-15 18:07:24|         163|          79|   NULL| 202

# 5.3.4: SQL with Spark

We need to first download and load taxi data. I will use the provided scripts to download and process data to parquet files

In [1]:
!bash /app/scripts/download_data.sh yellow 2020
!bash /app/scripts/download_data.sh green 2020

downloading https://github.com/DataTalksClub/nyc-tlc-data/releases/download//yellow/yellow_tripdata_2020-01.csv.gz to /app/data/raw/yellow/2020/01/yellow_tripdata_2020_01.csv.gz
--2026-03-01 18:46:11--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download//yellow/yellow_tripdata_2020-01.csv.gz
Resolving github.com (github.com)... 4.228.31.150
Connecting to github.com (github.com)|4.228.31.150|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/1bc3de9c-af03-46e3-baa6-c4475782682d?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-01T19%3A35%3A12Z&rscd=attachment%3B+filename%3Dyellow_tripdata_2020-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-01T18%3A34%3A49Z&ske=2026-03-01T19%3A35%3A12Z&sks=b&skv=2018-11-09&sig=b3kfBW4j4jgVfkyKOGvyEzZZ2W6OJ%2FGfZnLbazWxnn4%3D&jwt=eyJ0eXAiOiJKV